# Pandas Part 3 — Practical Advanced Pandas

This notebook is the third and final stage of the Pandas learning series.

It focuses on practical skills for **Data Science, Machine Learning, and Bioinformatics**.

## Topics
1. Advanced Data Cleaning
2. Feature Engineering
3. Advanced GroupBy
4. Advanced Merge & Join
5. Reshaping
6. Categorical Data
7. Time Series
8. Text Processing
9. Data Validation
10. Pandas + NumPy
11. Pandas for Bioinformatics

The goal is to understand **what a method does, why it is useful, and when to use it**.

In [1]:
import numpy as np
import pandas as pd

# Make DataFrame output easier to inspect.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

print("Pandas version:", pd.__version__)

Pandas version: 3.0.3


# 1. Advanced Data Cleaning

Real-world datasets commonly contain whitespace, inconsistent labels, invalid values, mixed formats, missing values, and duplicates.

A robust cleaning workflow should **detect problems before changing the data**.

In [2]:
raw = pd.DataFrame({
    "Patient_ID": [" P001 ", "P002", "p003", "P004", "P005"],
    "Sex": ["Male", " female ", "M", "Female", "male"],
    "Age": [45, 52, -3, 67, 250],
    "Tumor_Type": ["GBM", "gbm", "Glioblastoma", "GBM ", "glioblastoma"]
})

print(raw)

  Patient_ID       Sex  Age    Tumor_Type
0      P001       Male   45           GBM
1       P002   female    52           gbm
2       p003         M   -3  Glioblastoma
3       P004    Female   67          GBM 
4       P005      male  250  glioblastoma


## 1.1 Standardizing text

Use `.str.strip()` to remove surrounding whitespace and `.str.lower()` / `.str.upper()` to standardize case.

In [3]:
clean = raw.copy()

# Remove unnecessary spaces and standardize text case.
clean["Patient_ID"] = clean["Patient_ID"].str.strip().str.upper()
clean["Sex"] = clean["Sex"].str.strip().str.lower()
clean["Tumor_Type"] = clean["Tumor_Type"].str.strip().str.lower()

print(clean)

  Patient_ID     Sex  Age    Tumor_Type
0       P001    male   45           gbm
1       P002  female   52           gbm
2       P003       m   -3  glioblastoma
3       P004  female   67           gbm
4       P005    male  250  glioblastoma


## 1.2 `replace()`

`replace()` is useful for converting inconsistent labels into a common representation.

In [4]:
clean["Sex"] = clean["Sex"].replace({
    "m": "Male",
    "male": "Male",
    "f": "Female",
    "female": "Female"
})

clean["Tumor_Type"] = clean["Tumor_Type"].replace({
    "gbm": "Glioblastoma"
})

print(clean)

  Patient_ID     Sex  Age    Tumor_Type
0       P001    Male   45  Glioblastoma
1       P002  Female   52  Glioblastoma
2       P003    Male   -3  glioblastoma
3       P004  Female   67  Glioblastoma
4       P005    Male  250  glioblastoma


## 1.3 `map()`

`map()` applies a dictionary or function to each value. It is useful for direct mappings.

In [5]:
sex_code = {
    "Male": 1,
    "Female": 0
}

clean["Sex_Code"] = clean["Sex"].map(sex_code)

print(clean)

  Patient_ID     Sex  Age    Tumor_Type  Sex_Code
0       P001    Male   45  Glioblastoma         1
1       P002  Female   52  Glioblastoma         0
2       P003    Male   -3  glioblastoma         1
3       P004  Female   67  Glioblastoma         0
4       P005    Male  250  glioblastoma         1


## 1.4 `apply()`

`apply()` is useful for custom transformations when a vectorized operation is not convenient.

In [6]:
# Demonstration only: do not blindly correct invalid measurements in real research data.
clean["Age_Absolute"] = clean["Age"].apply(abs)

print(clean)

  Patient_ID     Sex  Age    Tumor_Type  Sex_Code  Age_Absolute
0       P001    Male   45  Glioblastoma         1            45
1       P002  Female   52  Glioblastoma         0            52
2       P003    Male   -3  glioblastoma         1             3
3       P004  Female   67  Glioblastoma         0            67
4       P005    Male  250  glioblastoma         1           250


## 1.5 Detecting invalid values

Do not silently repair suspicious biological data. First identify it, investigate it, and then decide what to do.

In [7]:
invalid_age = (clean["Age"] < 0) | (clean["Age"] > 120)

print("Invalid age records:")
print(clean[invalid_age])

Invalid age records:
  Patient_ID   Sex  Age    Tumor_Type  Sex_Code  Age_Absolute
2       P003  Male   -3  glioblastoma         1             3
4       P005  Male  250  glioblastoma         1           250


In [8]:
# After deciding that these values are invalid, mark them as missing.
clean.loc[invalid_age, "Age"] = np.nan

print(clean)

  Patient_ID     Sex   Age    Tumor_Type  Sex_Code  Age_Absolute
0       P001    Male  45.0  Glioblastoma         1            45
1       P002  Female  52.0  Glioblastoma         0            52
2       P003    Male   NaN  glioblastoma         1             3
3       P004  Female  67.0  Glioblastoma         0            67
4       P005    Male   NaN  glioblastoma         1           250


## 1.6 `where()` and `mask()`

- `where(condition)` keeps values where the condition is True.
- `mask(condition)` replaces values where the condition is True.

In [9]:
ages = pd.Series([25, 40, 150, 55, -2, 70])

# Keep only values inside the accepted range.
valid_ages = ages.where(ages.between(0, 120))

print("Original:")
print(ages)

print("\nValidated:")
print(valid_ages)

Original:
0     25
1     40
2    150
3     55
4     -2
5     70
dtype: int64

Validated:
0    25.0
1    40.0
2     NaN
3    55.0
4     NaN
5    70.0
dtype: float64


# 2. Feature Engineering

Feature engineering creates useful variables from existing data.

Examples include ratios, log transformations, age groups, binary indicators, and other domain-specific features.

In [10]:
patients = pd.DataFrame({
    "Patient_ID": ["P001", "P002", "P003", "P004", "P005"],
    "Age": [35, 52, 68, 44, 75],
    "Tumor_Volume": [12.5, 30.2, 18.4, 45.0, 22.1],
    "Expression": [2.5, 8.2, 15.0, 4.1, 20.5],
    "Survival_Days": [900, 500, 250, 720, 180]
})

print(patients)

  Patient_ID  Age  Tumor_Volume  Expression  Survival_Days
0       P001   35          12.5         2.5            900
1       P002   52          30.2         8.2            500
2       P003   68          18.4        15.0            250
3       P004   44          45.0         4.1            720
4       P005   75          22.1        20.5            180


In [11]:
# Create numerical features from existing columns.
patients["Volume_per_Age"] = patients["Tumor_Volume"] / patients["Age"]

# log1p(x) calculates log(1 + x), which is safe when x can be zero.
patients["Log_Expression"] = np.log1p(patients["Expression"])

print(patients)

  Patient_ID  Age  Tumor_Volume  Expression  Survival_Days  Volume_per_Age  \
0       P001   35          12.5         2.5            900        0.357143   
1       P002   52          30.2         8.2            500        0.580769   
2       P003   68          18.4        15.0            250        0.270588   
3       P004   44          45.0         4.1            720        1.022727   
4       P005   75          22.1        20.5            180        0.294667   

   Log_Expression  
0        1.252763  
1        2.219203  
2        2.772589  
3        1.629241  
4        3.068053  


## 2.1 `np.where()`

Use `np.where()` for a two-way condition.

In [12]:
patients["Older_Adult"] = np.where(
    patients["Age"] >= 65,
    "Yes",
    "No"
)

print(patients[["Age", "Older_Adult"]])

   Age Older_Adult
0   35          No
1   52          No
2   68         Yes
3   44          No
4   75         Yes


## 2.2 `np.select()`

Use `np.select()` when several conditions are needed.

In [13]:
conditions = [
    patients["Age"] < 40,
    patients["Age"].between(40, 64),
    patients["Age"] >= 65
]

choices = [
    "Young",
    "Middle-aged",
    "Older"
]

patients["Age_Group"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

print(patients[["Age", "Age_Group"]])

   Age    Age_Group
0   35        Young
1   52  Middle-aged
2   68        Older
3   44  Middle-aged
4   75        Older


In [14]:
# Ratio features can describe relationships between measurements.
patients["Expression_per_Volume"] = (
    patients["Expression"] / patients["Tumor_Volume"]
)

print(patients[["Expression", "Tumor_Volume", "Expression_per_Volume"]])

   Expression  Tumor_Volume  Expression_per_Volume
0         2.5          12.5               0.200000
1         8.2          30.2               0.271523
2        15.0          18.4               0.815217
3         4.1          45.0               0.091111
4        20.5          22.1               0.927602


# 3. Advanced GroupBy

`groupby()` divides data into groups.

The three concepts to remember are:

- `agg()` → summarize groups
- `transform()` → return group-level values aligned with original rows
- `apply()` → custom operation on each group

In [15]:
clinical = pd.DataFrame({
    "Treatment": ["A", "A", "B", "B", "A", "B"],
    "Sex": ["M", "F", "M", "F", "M", "F"],
    "Age": [45, 55, 62, 70, 50, 66],
    "Survival_Days": [800, 650, 500, 300, 720, 420]
})

print(clinical)

  Treatment Sex  Age  Survival_Days
0         A   M   45            800
1         A   F   55            650
2         B   M   62            500
3         B   F   70            300
4         A   M   50            720
5         B   F   66            420


In [16]:
summary = clinical.groupby("Treatment").agg(
    Mean_Age=("Age", "mean"),
    Median_Survival=("Survival_Days", "median"),
    Patient_Count=("Survival_Days", "size")
)

print(summary)

           Mean_Age  Median_Survival  Patient_Count
Treatment                                          
A              50.0            720.0              3
B              66.0            420.0              3


## 3.1 `transform()`

`transform()` keeps the same number of rows as the original data, making it ideal for adding group-level features.

In [17]:
clinical["Treatment_Mean_Survival"] = (
    clinical.groupby("Treatment")["Survival_Days"]
    .transform("mean")
)

clinical["Survival_vs_Treatment_Mean"] = (
    clinical["Survival_Days"]
    - clinical["Treatment_Mean_Survival"]
)

print(clinical)

  Treatment Sex  Age  Survival_Days  Treatment_Mean_Survival  \
0         A   M   45            800               723.333333   
1         A   F   55            650               723.333333   
2         B   M   62            500               406.666667   
3         B   F   70            300               406.666667   
4         A   M   50            720               723.333333   
5         B   F   66            420               406.666667   

   Survival_vs_Treatment_Mean  
0                   76.666667  
1                  -73.333333  
2                   93.333333  
3                 -106.666667  
4                   -3.333333  
5                   13.333333  


## 3.2 `filter()`

`filter()` keeps complete groups that satisfy a condition.

In [18]:
# Keep treatment groups containing at least three patients.
filtered = clinical.groupby("Treatment").filter(
    lambda group: len(group) >= 3
)

print(filtered)

  Treatment Sex  Age  Survival_Days  Treatment_Mean_Survival  \
0         A   M   45            800               723.333333   
1         A   F   55            650               723.333333   
2         B   M   62            500               406.666667   
3         B   F   70            300               406.666667   
4         A   M   50            720               723.333333   
5         B   F   66            420               406.666667   

   Survival_vs_Treatment_Mean  
0                   76.666667  
1                  -73.333333  
2                   93.333333  
3                 -106.666667  
4                   -3.333333  
5                   13.333333  


## 3.3 `apply()`

`apply()` is flexible, but specialized Pandas methods are usually preferable when available.

In [19]:
survival_range = clinical.groupby("Treatment")["Survival_Days"].apply(
    lambda group: group.max() - group.min()
)

print(survival_range)

Treatment
A    150
B    200
Name: Survival_Days, dtype: int64


### `agg()` vs `transform()` vs `apply()`

| Method | Main purpose |
|---|---|
| `agg()` | Group summary |
| `transform()` | Group-level value aligned to original rows |
| `apply()` | Custom group operation |

# 4. Advanced Merge & Join

Biological projects often store clinical, expression, mutation, and treatment information in separate tables.

Patient IDs are commonly used to connect them.

In [20]:
clinical = pd.DataFrame({
    "Patient_ID": ["P001", "P002", "P003", "P004"],
    "Age": [45, 52, 61, 70],
    "Survival_Days": [800, 500, 300, 200]
})

molecular = pd.DataFrame({
    "Patient_ID": ["P001", "P002", "P003", "P005"],
    "TP53": [1.2, 3.4, 2.1, 5.5],
    "EGFR": [8.1, 4.2, 6.0, 7.3]
})

print("Clinical:")
print(clinical)

print("\nMolecular:")
print(molecular)

Clinical:
  Patient_ID  Age  Survival_Days
0       P001   45            800
1       P002   52            500
2       P003   61            300
3       P004   70            200

Molecular:
  Patient_ID  TP53  EGFR
0       P001   1.2   8.1
1       P002   3.4   4.2
2       P003   2.1   6.0
3       P005   5.5   7.3


In [21]:
inner = clinical.merge(
    molecular,
    on="Patient_ID",
    how="inner"
)

print(inner)

  Patient_ID  Age  Survival_Days  TP53  EGFR
0       P001   45            800   1.2   8.1
1       P002   52            500   3.4   4.2
2       P003   61            300   2.1   6.0


In [22]:
left = clinical.merge(
    molecular,
    on="Patient_ID",
    how="left",
    indicator=True
)

print(left)

  Patient_ID  Age  Survival_Days  TP53  EGFR     _merge
0       P001   45            800   1.2   8.1       both
1       P002   52            500   3.4   4.2       both
2       P003   61            300   2.1   6.0       both
3       P004   70            200   NaN   NaN  left_only


## 4.1 Finding unmatched records

In [23]:
print("Clinical-only records:")
print(left[left["_merge"] == "left_only"])

print("\nMatched records:")
print(left[left["_merge"] == "both"])

Clinical-only records:
  Patient_ID  Age  Survival_Days  TP53  EGFR     _merge
3       P004   70            200   NaN   NaN  left_only

Matched records:
  Patient_ID  Age  Survival_Days  TP53  EGFR _merge
0       P001   45            800   1.2   8.1   both
1       P002   52            500   3.4   4.2   both
2       P003   61            300   2.1   6.0   both


## 4.2 `validate=`

`validate="one_to_one"` checks whether each key appears at most once in each table.

This is useful for detecting unexpected duplicate IDs.

In [24]:
validated = clinical.merge(
    molecular,
    on="Patient_ID",
    how="inner",
    validate="one_to_one"
)

print(validated)

  Patient_ID  Age  Survival_Days  TP53  EGFR
0       P001   45            800   1.2   8.1
1       P002   52            500   3.4   4.2
2       P003   61            300   2.1   6.0


## 4.3 Different key names

In [25]:
molecular_alt = molecular.rename(columns={"Patient_ID": "ID"})

merged = clinical.merge(
    molecular_alt,
    left_on="Patient_ID",
    right_on="ID",
    how="left"
)

print(merged)

  Patient_ID  Age  Survival_Days    ID  TP53  EGFR
0       P001   45            800  P001   1.2   8.1
1       P002   52            500  P002   3.4   4.2
2       P003   61            300  P003   2.1   6.0
3       P004   70            200   NaN   NaN   NaN


## 4.4 `concat()`

`concat()` can combine DataFrames by rows or columns.

In [26]:
part_a = clinical.iloc[:2]
part_b = clinical.iloc[2:]

combined_rows = pd.concat(
    [part_a, part_b],
    axis=0,
    ignore_index=True
)

print(combined_rows)

  Patient_ID  Age  Survival_Days
0       P001   45            800
1       P002   52            500
2       P003   61            300
3       P004   70            200


In [27]:
clinical_indexed = clinical.set_index("Patient_ID")

molecular_indexed = molecular.set_index("Patient_ID")

joined = clinical_indexed.join(
    molecular_indexed,
    how="left",
    lsuffix="_clinical",
    rsuffix="_molecular"
)

print(joined)

            Age  Survival_Days  TP53  EGFR
Patient_ID                                
P001         45            800   1.2   8.1
P002         52            500   3.4   4.2
P003         61            300   2.1   6.0
P004         70            200   NaN   NaN


# 5. Reshaping

Pandas can convert data between **wide** and **long** formats.

This is particularly useful for gene expression data.

In [28]:
expression_wide = pd.DataFrame({
    "Patient_ID": ["P001", "P002", "P003"],
    "TP53": [2.1, 4.5, 3.2],
    "EGFR": [8.0, 5.2, 7.1],
    "IDH1": [1.2, 0.8, 3.4]
})

print(expression_wide)

  Patient_ID  TP53  EGFR  IDH1
0       P001   2.1   8.0   1.2
1       P002   4.5   5.2   0.8
2       P003   3.2   7.1   3.4


In [29]:
expression_long = expression_wide.melt(
    id_vars="Patient_ID",
    var_name="Gene",
    value_name="Expression"
)

print(expression_long)

  Patient_ID  Gene  Expression
0       P001  TP53         2.1
1       P002  TP53         4.5
2       P003  TP53         3.2
3       P001  EGFR         8.0
4       P002  EGFR         5.2
5       P003  EGFR         7.1
6       P001  IDH1         1.2
7       P002  IDH1         0.8
8       P003  IDH1         3.4


In [30]:
expression_back = expression_long.pivot(
    index="Patient_ID",
    columns="Gene",
    values="Expression"
).reset_index()

print(expression_back)

Gene Patient_ID  EGFR  IDH1  TP53
0          P001   8.0   1.2   2.1
1          P002   5.2   0.8   4.5
2          P003   7.1   3.4   3.2


## 5.1 `pivot_table()`

Use `pivot_table()` when duplicate combinations exist and an aggregation is required.

In [31]:
measurements = pd.DataFrame({
    "Patient_ID": ["P001", "P001", "P002", "P002"],
    "Gene": ["TP53", "TP53", "TP53", "EGFR"],
    "Expression": [2.0, 4.0, 3.0, 8.0]
})

table = measurements.pivot_table(
    index="Patient_ID",
    columns="Gene",
    values="Expression",
    aggfunc="mean"
)

print(table)

Gene        EGFR  TP53
Patient_ID            
P001         NaN   3.0
P002         8.0   3.0


## 5.2 `explode()`

`explode()` converts list-like values into separate rows.

In [32]:
gene_lists = pd.DataFrame({
    "Patient_ID": ["P001", "P002"],
    "Mutations": [
        ["TP53", "EGFR"],
        ["IDH1", "ATRX", "TP53"]
    ]
})

print(gene_lists.explode("Mutations"))

  Patient_ID Mutations
0       P001      TP53
0       P001      EGFR
1       P002      IDH1
1       P002      ATRX
1       P002      TP53


## 5.3 `stack()` and `unstack()`

These operations are closely related to reshaping indexed data.

In [33]:
wide = expression_wide.set_index("Patient_ID")

stacked = wide.stack()

print("Stacked:")
print(stacked)

print("\nUnstacked:")
print(stacked.unstack())

Stacked:
Patient_ID      
P001        TP53    2.1
            EGFR    8.0
            IDH1    1.2
P002        TP53    4.5
            EGFR    5.2
            IDH1    0.8
P003        TP53    3.2
            EGFR    7.1
            IDH1    3.4
dtype: float64

Unstacked:
            TP53  EGFR  IDH1
Patient_ID                  
P001         2.1   8.0   1.2
P002         4.5   5.2   0.8
P003         3.2   7.1   3.4


# 6. Categorical Data

Categorical data contains a limited set of labels.

Examples include treatment groups, tumor stages, sex, and disease categories.

The `category` dtype can also reduce memory usage.

In [34]:
df_cat = pd.DataFrame({
    "Patient_ID": ["P001", "P002", "P003", "P004"],
    "Stage": ["II", "III", "I", "III"]
})

df_cat["Stage"] = df_cat["Stage"].astype("category")

print(df_cat.dtypes)
print(df_cat["Stage"].cat.categories)

Patient_ID         str
Stage         category
dtype: object
Index(['I', 'II', 'III'], dtype='str')


In [35]:
stage_order = ["I", "II", "III", "IV"]

df_cat["Stage"] = pd.Categorical(
    df_cat["Stage"],
    categories=stage_order,
    ordered=True
)

print(df_cat.sort_values("Stage"))

  Patient_ID Stage
2       P003     I
0       P001    II
1       P002   III
3       P004   III


In [36]:
# Rename category labels without changing the underlying category structure.
df_cat["Stage"] = df_cat["Stage"].cat.rename_categories({
    "I": "Stage I",
    "II": "Stage II",
    "III": "Stage III",
    "IV": "Stage IV"
})

print(df_cat)

  Patient_ID      Stage
0       P001   Stage II
1       P002  Stage III
2       P003    Stage I
3       P004  Stage III


# 7. Time Series

Important tools include:

- `to_datetime()`
- `.dt`
- `shift()`
- `diff()`
- `pct_change()`
- `rolling()`
- `expanding()`
- `ewm()`
- `resample()`

In [37]:
dates = pd.DataFrame({
    "Date": ["2026-01-01", "2026-01-02", "2026-01-03", "2026-01-04"],
    "Expression": [10, 12, 15, 18]
})

dates["Date"] = pd.to_datetime(dates["Date"])

print(dates)
print(dates.dtypes)

        Date  Expression
0 2026-01-01          10
1 2026-01-02          12
2 2026-01-03          15
3 2026-01-04          18
Date          datetime64[us]
Expression             int64
dtype: object


In [38]:
dates["Day"] = dates["Date"].dt.day
dates["Month"] = dates["Date"].dt.month
dates["Weekday"] = dates["Date"].dt.day_name()

print(dates)

        Date  Expression  Day  Month   Weekday
0 2026-01-01          10    1      1  Thursday
1 2026-01-02          12    2      1    Friday
2 2026-01-03          15    3      1  Saturday
3 2026-01-04          18    4      1    Sunday


In [39]:
dates["Previous"] = dates["Expression"].shift(1)
dates["Difference"] = dates["Expression"].diff()
dates["Percent_Change"] = dates["Expression"].pct_change()

print(dates)

        Date  Expression  Day  Month   Weekday  Previous  Difference  \
0 2026-01-01          10    1      1  Thursday       NaN         NaN   
1 2026-01-02          12    2      1    Friday      10.0         2.0   
2 2026-01-03          15    3      1  Saturday      12.0         3.0   
3 2026-01-04          18    4      1    Sunday      15.0         3.0   

   Percent_Change  
0             NaN  
1            0.20  
2            0.25  
3            0.20  


In [40]:
dates["Rolling_Mean"] = (
    dates["Expression"].rolling(window=2).mean()
)

print(dates)

        Date  Expression  Day  Month   Weekday  Previous  Difference  \
0 2026-01-01          10    1      1  Thursday       NaN         NaN   
1 2026-01-02          12    2      1    Friday      10.0         2.0   
2 2026-01-03          15    3      1  Saturday      12.0         3.0   
3 2026-01-04          18    4      1    Sunday      15.0         3.0   

   Percent_Change  Rolling_Mean  
0             NaN           NaN  
1            0.20          11.0  
2            0.25          13.5  
3            0.20          16.5  


In [41]:
daily = pd.DataFrame({
    "Date": pd.date_range("2026-01-01", periods=10, freq="D"),
    "Value": [10, 12, 15, 11, 18, 20, 17, 21, 23, 25]
}).set_index("Date")

weekly_mean = daily["Value"].resample("W").mean()

print(weekly_mean)

Date
2026-01-04    12.000000
2026-01-11    20.666667
Freq: W-SUN, Name: Value, dtype: float64


In [42]:
daily["Expanding_Mean"] = daily["Value"].expanding().mean()
daily["EWMA"] = daily["Value"].ewm(span=3).mean()

print(daily)

            Value  Expanding_Mean       EWMA
Date                                        
2026-01-01     10       10.000000  10.000000
2026-01-02     12       11.000000  11.333333
2026-01-03     15       12.333333  13.428571
2026-01-04     11       12.000000  12.133333
2026-01-05     18       13.200000  15.161290
2026-01-06     20       14.333333  17.619048
2026-01-07     17       14.714286  17.307087
2026-01-08     21       15.500000  19.160784
2026-01-09     23       16.333333  21.084149
2026-01-10     25       17.200000  23.043988


# 8. Text Processing

The `.str` accessor provides vectorized string operations.

These are useful for sample IDs, gene names, clinical labels, and metadata.

In [43]:
samples = pd.DataFrame({
    "Sample_ID": [" tumor_001 ", "Tumor_002", "normal_003", "tumor_004"],
    "Description": [
        "Primary GBM",
        "Recurrent GBM",
        "Normal tissue",
        "Primary tumor"
    ]
})

samples["Sample_ID"] = samples["Sample_ID"].str.strip().str.upper()

print(samples)

    Sample_ID    Description
0   TUMOR_001    Primary GBM
1   TUMOR_002  Recurrent GBM
2  NORMAL_003  Normal tissue
3   TUMOR_004  Primary tumor


In [44]:
samples["Is_Tumor"] = samples["Sample_ID"].str.startswith("TUMOR")

samples["Contains_GBM"] = (
    samples["Description"]
    .str.contains("GBM", case=False, na=False)
)

print(samples)

    Sample_ID    Description  Is_Tumor  Contains_GBM
0   TUMOR_001    Primary GBM      True          True
1   TUMOR_002  Recurrent GBM      True          True
2  NORMAL_003  Normal tissue     False         False
3   TUMOR_004  Primary tumor      True         False


In [45]:
samples["Description_Words"] = samples["Description"].str.split()

print(samples[["Description", "Description_Words"]])

     Description Description_Words
0    Primary GBM    [Primary, GBM]
1  Recurrent GBM  [Recurrent, GBM]
2  Normal tissue  [Normal, tissue]
3  Primary tumor  [Primary, tumor]


In [46]:
ids = pd.Series([
    "Patient_P001",
    "Patient_P002",
    "Patient_P103"
])

patient_numbers = ids.str.extract(
    r"Patient_(P\d+)",
    expand=False
)

print(patient_numbers)

0    P001
1    P002
2    P103
dtype: str


In [47]:
samples["Description"] = (
    samples["Description"]
    .str.replace("tumor", "neoplasm", case=False, regex=True)
)

print(samples)

    Sample_ID       Description  Is_Tumor  Contains_GBM Description_Words
0   TUMOR_001       Primary GBM      True          True    [Primary, GBM]
1   TUMOR_002     Recurrent GBM      True          True  [Recurrent, GBM]
2  NORMAL_003     Normal tissue     False         False  [Normal, tissue]
3   TUMOR_004  Primary neoplasm      True         False  [Primary, tumor]


# 9. Data Validation

Data validation asks whether the dataset has a trustworthy structure and plausible values.

A practical checklist:

- shape
- columns
- dtypes
- missing values
- duplicates
- unique categories
- numeric ranges
- identifier uniqueness
- logical consistency

In [48]:
validation = pd.DataFrame({
    "Patient_ID": ["P001", "P002", "P002", "P004"],
    "Age": [45, 60, 60, 130],
    "Survival_Days": [500, 300, -10, 800],
    "Sex": ["Male", "Female", "Female", "Unknown"]
})

print("Shape:", validation.shape)
print("\nDtypes:")
print(validation.dtypes)
print("\nMissing values:")
print(validation.isna().sum())
print("\nDuplicate rows:", validation.duplicated().sum())
print("\nUnique values:")
print(validation.nunique())

Shape: (4, 4)

Dtypes:
Patient_ID         str
Age              int64
Survival_Days    int64
Sex                str
dtype: object

Missing values:
Patient_ID       0
Age              0
Survival_Days    0
Sex              0
dtype: int64

Duplicate rows: 0

Unique values:
Patient_ID       3
Age              3
Survival_Days    4
Sex              3
dtype: int64


In [49]:
invalid_age = ~validation["Age"].between(0, 120)
invalid_survival = validation["Survival_Days"] < 0
invalid_sex = ~validation["Sex"].isin({"Male", "Female"})

print("Invalid ages:")
print(validation[invalid_age])

print("\nInvalid survival values:")
print(validation[invalid_survival])

print("\nInvalid sex values:")
print(validation[invalid_sex])

Invalid ages:
  Patient_ID  Age  Survival_Days      Sex
3       P004  130            800  Unknown

Invalid survival values:
  Patient_ID  Age  Survival_Days     Sex
2       P002   60            -10  Female

Invalid sex values:
  Patient_ID  Age  Survival_Days      Sex
3       P004  130            800  Unknown


In [50]:
duplicate_ids = validation[
    validation["Patient_ID"].duplicated(keep=False)
]

print("Duplicate Patient_ID records:")
print(duplicate_ids)

Duplicate Patient_ID records:
  Patient_ID  Age  Survival_Days     Sex
1       P002   60            300  Female
2       P002   60            -10  Female


### Validation workflow

```text
Raw Data
   ↓
Inspect structure
   ↓
Check data types
   ↓
Check missing values
   ↓
Check duplicates
   ↓
Check categories
   ↓
Check numeric ranges
   ↓
Check IDs
   ↓
Check logical consistency
   ↓
Clean / investigate / reject
```

# 10. Pandas + NumPy

A useful mental model:

- **NumPy** → numerical arrays and vectorized numerical computation
- **Pandas** → labeled tabular data and data manipulation

In [51]:
data = pd.DataFrame({
    "Age": [30, 45, 70, 55],
    "Expression": [2.0, 5.0, 12.0, 8.0]
})

# Convert selected columns into a NumPy array.
X = data[["Age", "Expression"]].to_numpy()

print(X)
print(type(X))

[[30.  2.]
 [45.  5.]
 [70. 12.]
 [55.  8.]]
<class 'numpy.ndarray'>


In [52]:
# NumPy can operate directly on Pandas columns.
data["Log_Expression"] = np.log1p(data["Expression"])

data["Age_Group"] = np.select(
    [
        data["Age"] < 40,
        data["Age"].between(40, 64),
        data["Age"] >= 65
    ],
    [
        "Young",
        "Middle-aged",
        "Older"
    ],
    default="Unknown"
)

print(data)

   Age  Expression  Log_Expression    Age_Group
0   30         2.0        1.098612        Young
1   45         5.0        1.791759  Middle-aged
2   70        12.0        2.564949        Older
3   55         8.0        2.197225  Middle-aged


## 10.1 Vectorization

Vectorized operations are generally shorter, clearer, and often faster than explicit Python loops.

In [53]:
# Vectorized calculation across the entire column.
data["Expression_After_Offset"] = data["Expression"] + 2

print(data)

   Age  Expression  Log_Expression    Age_Group  Expression_After_Offset
0   30         2.0        1.098612        Young                      4.0
1   45         5.0        1.791759  Middle-aged                      7.0
2   70        12.0        2.564949        Older                     14.0
3   55         8.0        2.197225  Middle-aged                     10.0


# 11. Pandas for Bioinformatics

Now we connect Pandas to a realistic biological data workflow.

We will use three conceptual tables:

1. Clinical data
2. Gene expression data
3. Mutation data

The goal is to build an integrated patient-level dataset.

In [54]:
clinical = pd.DataFrame({
    "Patient_ID": ["P001", "P002", "P003", "P004", "P005"],
    "Age": [45, 52, 61, 70, 39],
    "Sex": ["Male", "Female", "Male", "Female", "Male"],
    "Survival_Days": [800, 500, 300, 200, 950],
    "Event": [1, 1, 1, 1, 0]
})

expression = pd.DataFrame({
    "Patient_ID": ["P001", "P002", "P003", "P004", "P005"],
    "TP53": [2.1, 4.5, 3.2, 8.0, 1.5],
    "EGFR": [8.0, 5.2, 7.1, 9.2, 4.0],
    "IDH1": [1.2, 0.8, 3.4, 0.5, 4.2]
})

mutation = pd.DataFrame({
    "Patient_ID": ["P001", "P002", "P003", "P004", "P005"],
    "TP53_Mutated": [1, 0, 1, 1, 0],
    "EGFR_Mutated": [0, 1, 0, 1, 0],
    "IDH1_Mutated": [0, 1, 1, 0, 1]
})

print("Clinical:")
print(clinical)

print("\nExpression:")
print(expression)

print("\nMutation:")
print(mutation)

Clinical:
  Patient_ID  Age     Sex  Survival_Days  Event
0       P001   45    Male            800      1
1       P002   52  Female            500      1
2       P003   61    Male            300      1
3       P004   70  Female            200      1
4       P005   39    Male            950      0

Expression:
  Patient_ID  TP53  EGFR  IDH1
0       P001   2.1   8.0   1.2
1       P002   4.5   5.2   0.8
2       P003   3.2   7.1   3.4
3       P004   8.0   9.2   0.5
4       P005   1.5   4.0   4.2

Mutation:
  Patient_ID  TP53_Mutated  EGFR_Mutated  IDH1_Mutated
0       P001             1             0             0
1       P002             0             1             1
2       P003             1             0             1
3       P004             1             1             0
4       P005             0             0             1


In [55]:
integrated = clinical.merge(
    expression,
    on="Patient_ID",
    how="inner",
    validate="one_to_one"
)

print(integrated)

  Patient_ID  Age     Sex  Survival_Days  Event  TP53  EGFR  IDH1
0       P001   45    Male            800      1   2.1   8.0   1.2
1       P002   52  Female            500      1   4.5   5.2   0.8
2       P003   61    Male            300      1   3.2   7.1   3.4
3       P004   70  Female            200      1   8.0   9.2   0.5
4       P005   39    Male            950      0   1.5   4.0   4.2


In [56]:
integrated = integrated.merge(
    mutation,
    on="Patient_ID",
    how="left",
    validate="one_to_one"
)

print(integrated)

  Patient_ID  Age     Sex  Survival_Days  Event  TP53  EGFR  IDH1  \
0       P001   45    Male            800      1   2.1   8.0   1.2   
1       P002   52  Female            500      1   4.5   5.2   0.8   
2       P003   61    Male            300      1   3.2   7.1   3.4   
3       P004   70  Female            200      1   8.0   9.2   0.5   
4       P005   39    Male            950      0   1.5   4.0   4.2   

   TP53_Mutated  EGFR_Mutated  IDH1_Mutated  
0             1             0             0  
1             0             1             1  
2             1             0             1  
3             1             1             0  
4             0             0             1  


## 11.1 Filtering expression data

Filtering can identify patients above or below a gene-expression threshold.

In [57]:
high_egfr = integrated[
    integrated["EGFR"] > integrated["EGFR"].median()
]

print(high_egfr)

  Patient_ID  Age     Sex  Survival_Days  Event  TP53  EGFR  IDH1  \
0       P001   45    Male            800      1   2.1   8.0   1.2   
3       P004   70  Female            200      1   8.0   9.2   0.5   

   TP53_Mutated  EGFR_Mutated  IDH1_Mutated  
0             1             0             0  
3             1             1             0  


## 11.2 Gene-level analysis

Long format makes gene-level grouping straightforward.

In [58]:
gene_expression = integrated[
    ["Patient_ID", "TP53", "EGFR", "IDH1"]
].melt(
    id_vars="Patient_ID",
    var_name="Gene",
    value_name="Expression"
)

gene_summary = gene_expression.groupby("Gene").agg(
    Mean_Expression=("Expression", "mean"),
    Median_Expression=("Expression", "median"),
    Patient_Count=("Patient_ID", "nunique")
)

print(gene_summary)

      Mean_Expression  Median_Expression  Patient_Count
Gene                                                   
EGFR             6.70                7.1              5
IDH1             2.02                1.2              5
TP53             3.86                3.2              5


## 11.3 Patient-level feature engineering

In [59]:
integrated["Log_EGFR"] = np.log1p(integrated["EGFR"])

integrated["Age_Group"] = np.select(
    [
        integrated["Age"] < 40,
        integrated["Age"].between(40, 64),
        integrated["Age"] >= 65
    ],
    [
        "Young",
        "Middle-aged",
        "Older"
    ],
    default="Unknown"
)

integrated["Mutation_Count"] = integrated[
    ["TP53_Mutated", "EGFR_Mutated", "IDH1_Mutated"]
].sum(axis=1)

print(integrated)

  Patient_ID  Age     Sex  Survival_Days  Event  TP53  EGFR  IDH1  \
0       P001   45    Male            800      1   2.1   8.0   1.2   
1       P002   52  Female            500      1   4.5   5.2   0.8   
2       P003   61    Male            300      1   3.2   7.1   3.4   
3       P004   70  Female            200      1   8.0   9.2   0.5   
4       P005   39    Male            950      0   1.5   4.0   4.2   

   TP53_Mutated  EGFR_Mutated  IDH1_Mutated  Log_EGFR    Age_Group  \
0             1             0             0  2.197225  Middle-aged   
1             0             1             1  1.824549  Middle-aged   
2             1             0             1  2.091864  Middle-aged   
3             1             1             0  2.322388        Older   
4             0             0             1  1.609438        Young   

   Mutation_Count  
0               1  
1               2  
2               2  
3               2  
4               1  


## 11.4 Exploratory correlation

Correlation can be useful for exploratory analysis, but it does **not** prove causation.

Also, a simple correlation with survival time is not a replacement for proper survival analysis when censoring is present.

In [60]:
numeric = integrated[
    ["Age", "TP53", "EGFR", "IDH1", "Survival_Days"]
]

print("Correlation matrix:")
print(numeric.corr())

Correlation matrix:
                    Age      TP53      EGFR      IDH1  Survival_Days
Age            1.000000  0.872811  0.717280 -0.454448      -0.977549
TP53           0.872811  1.000000  0.609240 -0.678567      -0.816783
EGFR           0.717280  0.609240  1.000000 -0.597403      -0.615399
IDH1          -0.454448 -0.678567 -0.597403  1.000000       0.428372
Survival_Days -0.977549 -0.816783 -0.615399  0.428372       1.000000


## 11.5 Preparing `X`, `y`, and the event indicator

This is the bridge from Pandas to Machine Learning.

For ordinary regression/classification, `X` and `y` may be enough. For survival analysis, retain both survival time and the event/censoring indicator.

In [61]:
X = integrated[
    ["Age", "TP53", "EGFR", "IDH1"]
].copy()

y = integrated["Survival_Days"].copy()
event = integrated["Event"].copy()

print("X:")
print(X)

print("\ny:")
print(y)

print("\nEvent:")
print(event)

X:
   Age  TP53  EGFR  IDH1
0   45   2.1   8.0   1.2
1   52   4.5   5.2   0.8
2   61   3.2   7.1   3.4
3   70   8.0   9.2   0.5
4   39   1.5   4.0   4.2

y:
0    800
1    500
2    300
3    200
4    950
Name: Survival_Days, dtype: int64

Event:
0    1
1    1
2    1
3    1
4    0
Name: Event, dtype: int64


# Exercises

### Exercise 1 — Advanced Cleaning
Create a dataset with inconsistent sex labels and standardize them.

### Exercise 2 — Feature Engineering
Create an age-group feature and a log-transformed expression feature.

### Exercise 3 — GroupBy
Calculate mean and median survival for each treatment group.

### Exercise 4 — Merge
Create clinical and molecular tables and perform inner and left merges.

### Exercise 5 — Reshaping
Convert a wide gene-expression table into long format with `melt()`.

### Exercise 6 — Validation
Find invalid ages, negative survival values, duplicated patient IDs, and unexpected categories.

### Exercise 7 — Bioinformatics
Merge clinical, expression, and mutation tables and create a mutation-count feature.

# Cheat Sheet

| Task | Main tool |
|---|---|
| Clean whitespace | `.str.strip()` |
| Standardize case | `.str.lower()`, `.str.upper()` |
| Replace labels | `replace()` |
| Map values | `map()` |
| Custom transformation | `apply()` |
| Conditional values | `np.where()` |
| Multiple conditions | `np.select()` |
| Group summary | `groupby().agg()` |
| Group-aligned feature | `groupby().transform()` |
| Group filtering | `groupby().filter()` |
| Combine tables | `merge()` |
| Index-based join | `join()` |
| Stack tables | `concat()` |
| Wide → Long | `melt()` |
| Long → Wide | `pivot()` |
| Aggregated pivot | `pivot_table()` |
| List → Rows | `explode()` |
| Categories | `astype("category")` |
| Ordered categories | `pd.Categorical()` |
| Parse dates | `pd.to_datetime()` |
| Date components | `.dt` |
| Time aggregation | `resample()` |
| Previous value | `shift()` |
| Difference | `diff()` |
| Percentage change | `pct_change()` |
| Moving statistic | `rolling()` |
| Text search | `.str.contains()` |
| Text extraction | `.str.extract()` |
| Validation | `isna()`, `duplicated()`, `nunique()` |
| Correlation | `corr()` |
| NumPy conversion | `.to_numpy()` |

# Golden Rules

1. Inspect data before modifying it.
2. Never blindly overwrite suspicious biological measurements.
3. Validate patient IDs before merging tables.
4. Use `validate=` in important merges when appropriate.
5. Use `agg()` for summaries and `transform()` for row-aligned group features.
6. Prefer vectorized operations over unnecessary loops.
7. Keep raw and cleaned data separate.
8. Document important cleaning decisions.
9. Check for data leakage before Machine Learning.
10. Keep survival time and event/censoring information together for survival problems.
11. Use Pandas for tabular manipulation and NumPy for numerical array operations.
12. Do not try to memorize every Pandas method; learn how to solve data problems efficiently.

More parts will be added in the future